# Wikidata Article Identifier Enrichment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_Article_Identifier_Enrichment.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook enriches Wikidata scholarly articles with external database identifiers by querying multiple free APIs. Given articles with DOIs already in Wikidata, it looks up each DOI across OpenAlex, Semantic Scholar, Fatcat (Internet Archive), and PubMed to discover additional identifiers, then generates QuickStatements to add them with proper source attribution.

## Key Features

- **Multi-Source Lookup**: Queries 4 free APIs (OpenAlex, Semantic Scholar, Fatcat, PubMed)
- **No API Keys Required**: Uses only free, open endpoints with polite rate limiting
- **Batch Processing**: OpenAlex supports 50 DOIs per request for efficiency
- **Gap Analysis**: Shows which identifiers are already present vs. missing
- **Flexible Input**: Query by journal, upload DOI list, or use CSV file
- **Source Attribution**: Every identifier includes stated-in reference
- **Scholarly Endpoint Aware**: Uses `query-scholarly.wikidata.org` (required since May 2025)

## Identifiers Added

| Property | Identifier | Source API |
|----------|------------|------------|
| P10283 | OpenAlex ID | OpenAlex |
| P4011 | Semantic Scholar paper ID | Semantic Scholar |
| P8608 | Fatcat ID | Fatcat |
| P698 | PubMed ID | OpenAlex, PubMed |
| P932 | PMC ID | OpenAlex |
| P6366 | Microsoft Academic ID | OpenAlex (legacy) |

## Workflow

1. **Select Input**: Choose journal(s), paste DOIs, or upload CSV
2. **Query Wikidata**: Find articles and check existing identifiers
3. **Lookup APIs**: Query OpenAlex, Semantic Scholar, Fatcat, and PubMed
4. **Review**: See which identifiers were discovered and gap analysis
5. **Generate**: Create QuickStatements with source attribution
6. **Export**: Download QuickStatements file and summary CSV

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
!pip install requests pandas ipywidgets -q

import requests
import pandas as pd
import re
import time
import json
import urllib.parse
from datetime import datetime
from collections import defaultdict
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO
import os

print("✓ Setup complete.")

## Configuration

*Define API endpoints, property mappings, and rate limiting settings.*

**CRITICAL**: Since May 2025, scholarly articles are ONLY on `query-scholarly.wikidata.org`. The main Wikidata endpoint no longer contains scholarly articles!

In [ ]:
# =============================================================================
# WIKIDATA ENDPOINTS
# =============================================================================
# Since May 2025, scholarly articles are ONLY on the scholarly endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"
MAIN_ENDPOINT = "https://query.wikidata.org/sparql"

# =============================================================================
# EXTERNAL API ENDPOINTS
# =============================================================================
OPENALEX_API = "https://api.openalex.org/works"
SEMANTIC_SCHOLAR_API = "https://api.semanticscholar.org/graph/v1/paper"
FATCAT_API = "https://api.fatcat.wiki/v0/release/lookup"
PUBMED_ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

# =============================================================================
# WIKIDATA PROPERTY IDs FOR IDENTIFIERS
# =============================================================================
IDENTIFIER_PROPERTIES = {
    'openalex': 'P10283',      # OpenAlex ID
    'semantic_scholar': 'P4011', # Semantic Scholar paper ID
    'fatcat': 'P8608',          # Fatcat ID
    'pmid': 'P698',             # PubMed ID
    'pmcid': 'P932',            # PubMed Central ID
    'mag': 'P6366',             # Microsoft Academic ID (deprecated but still useful)
}

# Human-readable names
IDENTIFIER_NAMES = {
    'openalex': 'OpenAlex ID',
    'semantic_scholar': 'Semantic Scholar ID',
    'fatcat': 'Fatcat ID',
    'pmid': 'PubMed ID',
    'pmcid': 'PMC ID',
    'mag': 'Microsoft Academic ID',
}

# =============================================================================
# WIKIDATA Q-IDs FOR SOURCE ATTRIBUTION
# =============================================================================
SOURCE_QIDS = {
    'openalex': 'Q107507940',      # OpenAlex
    'semantic_scholar': 'Q22908627', # Semantic Scholar
    'fatcat': 'Q145624554',         # Fatcat
    'pubmed': 'Q180686',            # PubMed
}

# =============================================================================
# RATE LIMITING SETTINGS
# =============================================================================
RATE_LIMITS = {
    'openalex': 1.0,        # 1 second between requests
    'semantic_scholar': 3.5, # 100 req/5 min = ~3 sec minimum
    'fatcat': 1.0,          # Generous limits
    'pubmed': 0.35,         # 3 req/sec without API key
    'wikidata': 0.5,        # Be nice to Wikidata
}

# =============================================================================
# USER CONFIGURATION
# =============================================================================
# Add your email for OpenAlex "polite pool" (faster rate limits)
USER_EMAIL = "your.email@example.com"  # <-- CHANGE THIS

USER_AGENT = "WikidataIdentifierEnrichment/1.0 (Wikidata scholarly metadata project; https://www.mattartz.me/)"

# =============================================================================
# STYLING
# =============================================================================
COLORS = {
    'bg_primary': '#E7ECEF',
    'text_primary': '#274C77',
    'interactive': '#6096BA',
    'bg_secondary': '#A3CEF1',
    'neutral': '#8B8C89',
    'success': '#28a745',
    'warning': '#ffc107',
    'error': '#dc3545'
}

CONTAINER_STYLE = f"""
    background-color: {COLORS['bg_primary']};
    border-left: 5px solid {COLORS['text_primary']};
    border-radius: 10px;
    padding: 15px;
    margin: 10px 0;
"""

print("✓ Configuration loaded")
print(f"  Scholarly endpoint: {SCHOLARLY_ENDPOINT}")
print(f"  OpenAlex API: {OPENALEX_API}")
print(f"  Semantic Scholar API: {SEMANTIC_SCHOLAR_API}")
print(f"  Fatcat API: {FATCAT_API}")
print(f"  PubMed E-utilities: {PUBMED_ESEARCH}")
print(f"\n⚠️  Remember to set USER_EMAIL for OpenAlex polite pool!")

## Helper Functions: DOI Utilities

*Functions for DOI parsing, cleaning, and validation.*

In [ ]:
def clean_doi(doi_input):
    """
    Clean and normalize a DOI string.
    Handles URLs, doi: prefixes, and whitespace.
    Returns None if invalid.
    """
    if not doi_input or not isinstance(doi_input, str):
        return None
    
    doi_input = str(doi_input).strip()
    
    # Handle empty strings
    if not doi_input:
        return None
    
    # Extract DOI from various URL formats
    patterns = [
        r'https?://(?:dx\.)?doi\.org/(.+)',
        r'doi:(.+)',
        r'DOI:(.+)',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    
    # Validate DOI format (must start with 10.)
    doi_input = doi_input.strip()
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input
    
    return None


def normalize_doi_for_wikidata(doi):
    """
    Normalize DOI for Wikidata comparison.
    Wikidata stores DOIs in UPPERCASE.
    """
    if not doi:
        return None
    return doi.upper()


def extract_dois_from_text(text):
    """
    Extract all DOIs from a block of text.
    Returns list of cleaned DOIs.
    """
    # Match DOI patterns in text
    pattern = r'10\.\d{4,}/[^\s"<>]+'
    matches = re.findall(pattern, text)
    
    # Clean and deduplicate
    dois = []
    seen = set()
    for match in matches:
        # Remove trailing punctuation
        cleaned = re.sub(r'[.,;:)\]]+$', '', match)
        cleaned = clean_doi(cleaned)
        if cleaned and cleaned.lower() not in seen:
            dois.append(cleaned)
            seen.add(cleaned.lower())
    
    return dois


print("✓ DOI utilities loaded")

# Test
test_dois = [
    "10.1111/test.123",
    "https://doi.org/10.1111/test.456",
    "doi:10.1111/test.789",
    "invalid",
]
print("\nDOI cleaning test:")
for d in test_dois:
    print(f"  '{d}' → '{clean_doi(d)}'")

## Helper Functions: Wikidata Queries

*Functions for querying Wikidata's scholarly endpoint.*

In [ ]:
def query_wikidata(sparql, endpoint=SCHOLARLY_ENDPOINT, max_retries=3):
    """
    Execute a SPARQL query against Wikidata.
    Returns list of result bindings or empty list on error.
    """
    headers = {
        'User-Agent': USER_AGENT,
        'Accept': 'application/sparql-results+json'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(
                endpoint,
                params={'query': sparql, 'format': 'json'},
                headers=headers,
                timeout=60
            )
            response.raise_for_status()
            return response.json().get('results', {}).get('bindings', [])
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait = (attempt + 1) * 5
                print(f"  ⚠️ Query failed, retrying in {wait}s: {e}")
                time.sleep(wait)
            else:
                print(f"  ❌ Query failed after {max_retries} attempts: {e}")
                return []


def get_articles_by_journal(journal_qid, limit=5000):
    """
    Get all scholarly articles for a journal with their DOIs and existing identifiers.
    Returns DataFrame with QID, DOI, and existing identifier columns.
    """
    # Build property list for existing identifiers
    prop_vars = []
    prop_optionals = []
    for key, prop in IDENTIFIER_PROPERTIES.items():
        var = f"?{key}"
        prop_vars.append(var)
        prop_optionals.append(f"OPTIONAL {{ ?article wdt:{prop} {var} . }}")
    
    sparql = f"""
    SELECT ?article ?doi {' '.join(prop_vars)}
    WHERE {{
      ?article wdt:P1433 wd:{journal_qid} .
      ?article wdt:P356 ?doi .
      {chr(10).join(prop_optionals)}
    }}
    LIMIT {limit}
    """
    
    results = query_wikidata(sparql)
    
    if not results:
        return pd.DataFrame()
    
    rows = []
    for r in results:
        row = {
            'qid': r['article']['value'].split('/')[-1],
            'doi': r['doi']['value'],
        }
        for key in IDENTIFIER_PROPERTIES.keys():
            row[f'existing_{key}'] = r.get(key, {}).get('value', '')
        rows.append(row)
    
    return pd.DataFrame(rows)


def lookup_article_by_doi(doi):
    """
    Look up a single article by DOI.
    Returns dict with QID and existing identifiers, or None if not found.
    """
    doi_upper = normalize_doi_for_wikidata(doi)
    
    # Build property optionals
    prop_vars = []
    prop_optionals = []
    for key, prop in IDENTIFIER_PROPERTIES.items():
        var = f"?{key}"
        prop_vars.append(var)
        prop_optionals.append(f"OPTIONAL {{ ?article wdt:{prop} {var} . }}")
    
    sparql = f"""
    SELECT ?article {' '.join(prop_vars)}
    WHERE {{
      ?article wdt:P356 "{doi_upper}" .
      {chr(10).join(prop_optionals)}
    }}
    LIMIT 1
    """
    
    results = query_wikidata(sparql)
    
    if not results:
        return None
    
    r = results[0]
    result = {
        'qid': r['article']['value'].split('/')[-1],
        'doi': doi,
    }
    for key in IDENTIFIER_PROPERTIES.keys():
        result[f'existing_{key}'] = r.get(key, {}).get('value', '')
    
    return result


def batch_lookup_articles_by_doi(dois, progress_callback=None):
    """
    Look up multiple articles by DOI.
    Returns DataFrame with QID, DOI, and existing identifiers.
    """
    rows = []
    total = len(dois)
    
    for i, doi in enumerate(dois):
        if progress_callback:
            progress_callback(i + 1, total, f"Looking up {doi[:30]}...")
        
        result = lookup_article_by_doi(doi)
        if result:
            rows.append(result)
        
        time.sleep(RATE_LIMITS['wikidata'])
    
    return pd.DataFrame(rows)


print("✓ Wikidata query functions loaded")

## Helper Functions: OpenAlex API

*Functions for querying OpenAlex. Supports batch lookups of up to 50 DOIs per request.*

In [ ]:
def query_openalex_batch(dois, email=USER_EMAIL):
    """
    Query OpenAlex for multiple DOIs (max 50 per request).
    Returns dict mapping DOI -> identifier dict.
    
    OpenAlex returns:
    - openalex: OpenAlex ID (e.g., "W2741809807")
    - pmid: PubMed ID (numeric string)
    - pmcid: PMC ID (e.g., "PMC1234567")
    - mag: Microsoft Academic ID (numeric)
    """
    if not dois:
        return {}
    
    # OpenAlex accepts up to 50 DOIs per request
    if len(dois) > 50:
        raise ValueError("Maximum 50 DOIs per batch")
    
    # Build filter string
    doi_filter = "|".join(dois)
    
    params = {
        'filter': f'doi:{doi_filter}',
        'per_page': 50,
    }
    
    # Add email for polite pool
    if email and email != "your.email@example.com":
        params['mailto'] = email
    
    headers = {'User-Agent': USER_AGENT}
    
    try:
        response = requests.get(
            OPENALEX_API,
            params=params,
            headers=headers,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"  ⚠️ OpenAlex request failed: {e}")
        return {}
    
    results = {}
    for work in data.get('results', []):
        ids = work.get('ids', {})
        
        # Extract DOI (normalize for matching)
        doi_url = ids.get('doi', '')
        if doi_url:
            doi = doi_url.replace('https://doi.org/', '')
        else:
            continue
        
        # Extract identifiers
        result = {}
        
        # OpenAlex ID (extract from URL)
        openalex_url = ids.get('openalex', '')
        if openalex_url:
            result['openalex'] = openalex_url.split('/')[-1]
        
        # PubMed ID (extract from URL)
        pmid_url = ids.get('pmid', '')
        if pmid_url:
            result['pmid'] = pmid_url.split('/')[-1]
        
        # PMC ID (extract from URL)
        pmcid_url = ids.get('pmcid', '')
        if pmcid_url:
            # Extract just the ID part (e.g., "PMC1234567" from URL)
            pmcid = pmcid_url.split('/')[-1]
            # Remove "PMC" prefix if present for Wikidata format
            result['pmcid'] = pmcid.replace('PMC', '')
        
        # Microsoft Academic ID (deprecated but sometimes available)
        mag_id = ids.get('mag', '')
        if mag_id:
            result['mag'] = str(mag_id)
        
        results[doi.lower()] = result
    
    return results


def query_openalex_all(dois, email=USER_EMAIL, progress_callback=None):
    """
    Query OpenAlex for all DOIs, handling batching automatically.
    Returns dict mapping DOI -> identifier dict.
    """
    all_results = {}
    total = len(dois)
    batch_size = 50
    
    for i in range(0, total, batch_size):
        batch = dois[i:i + batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = (total + batch_size - 1) // batch_size
        
        if progress_callback:
            progress_callback(i + len(batch), total, f"OpenAlex batch {batch_num}/{total_batches}")
        
        results = query_openalex_batch(batch, email)
        all_results.update(results)
        
        # Rate limiting
        if i + batch_size < total:
            time.sleep(RATE_LIMITS['openalex'])
    
    return all_results


print("✓ OpenAlex functions loaded")
print(f"  Batch size: 50 DOIs per request")
print(f"  Returns: OpenAlex ID, PubMed ID, PMC ID, MAG ID")

## Helper Functions: Semantic Scholar API

*Functions for querying Semantic Scholar. Returns S2 paper IDs and any additional identifiers.*

In [ ]:
def query_semantic_scholar(doi):
    """
    Query Semantic Scholar for a single DOI.
    Returns dict with identifiers or None if not found.
    """
    url = f"{SEMANTIC_SCHOLAR_API}/DOI:{doi}"
    params = {'fields': 'paperId,externalIds'}
    headers = {'User-Agent': USER_AGENT}
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        
        if response.status_code == 404:
            return None
        
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        # Don't print for 404s (common for papers not in S2)
        if not isinstance(e, requests.exceptions.HTTPError) or e.response.status_code != 404:
            pass  # Silently skip errors
        return None
    
    result = {}
    
    # Semantic Scholar paper ID
    paper_id = data.get('paperId')
    if paper_id:
        result['semantic_scholar'] = paper_id
    
    # External IDs (may include PMID, ArXiv, etc.)
    external_ids = data.get('externalIds', {})
    
    if external_ids.get('PubMed'):
        result['pmid'] = str(external_ids['PubMed'])
    
    if external_ids.get('ArXiv'):
        result['arxiv'] = external_ids['ArXiv']
    
    return result if result else None


def query_semantic_scholar_all(dois, progress_callback=None):
    """
    Query Semantic Scholar for all DOIs.
    Returns dict mapping DOI -> identifier dict.
    
    Note: S2 has strict rate limits (100 req/5 min without key).
    This function uses conservative delays.
    """
    results = {}
    total = len(dois)
    
    for i, doi in enumerate(dois):
        if progress_callback:
            progress_callback(i + 1, total, f"Semantic Scholar: {doi[:30]}...")
        
        result = query_semantic_scholar(doi)
        if result:
            results[doi.lower()] = result
        
        # Rate limiting (be conservative)
        if i < total - 1:
            time.sleep(RATE_LIMITS['semantic_scholar'])
    
    return results


print("✓ Semantic Scholar functions loaded")
print(f"  Rate limit: {RATE_LIMITS['semantic_scholar']}s between requests")
print(f"  Returns: Semantic Scholar paper ID, PubMed ID (if available)")

## Helper Functions: Fatcat API

*Functions for querying Internet Archive's Fatcat database.*

In [ ]:
def query_fatcat(doi):
    """
    Query Fatcat for a single DOI.
    Returns dict with Fatcat ID or None if not found.
    """
    params = {'doi': doi}
    headers = {'User-Agent': USER_AGENT}
    
    try:
        response = requests.get(FATCAT_API, params=params, headers=headers, timeout=30)
        
        if response.status_code == 404:
            return None
        
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        return None
    
    # Fatcat returns release info
    fatcat_id = data.get('ident')
    if fatcat_id:
        return {'fatcat': fatcat_id}
    
    return None


def query_fatcat_all(dois, progress_callback=None):
    """
    Query Fatcat for all DOIs.
    Returns dict mapping DOI -> identifier dict.
    """
    results = {}
    total = len(dois)
    
    for i, doi in enumerate(dois):
        if progress_callback:
            progress_callback(i + 1, total, f"Fatcat: {doi[:30]}...")
        
        result = query_fatcat(doi)
        if result:
            results[doi.lower()] = result
        
        # Rate limiting
        if i < total - 1:
            time.sleep(RATE_LIMITS['fatcat'])
    
    return results


print("✓ Fatcat functions loaded")
print(f"  Rate limit: {RATE_LIMITS['fatcat']}s between requests")
print(f"  Returns: Fatcat release ID")

## Helper Functions: PubMed E-utilities

*Functions for querying PubMed to verify/find PMIDs that other sources may have missed.*

In [ ]:
def query_pubmed(doi):
    """
    Query PubMed for a single DOI.
    Returns dict with PubMed ID or None if not found.
    """
    params = {
        'db': 'pubmed',
        'term': f'{doi}[doi]',
        'retmode': 'json',
    }
    headers = {'User-Agent': USER_AGENT}
    
    try:
        response = requests.get(PUBMED_ESEARCH, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        return None
    
    # Check for results
    esearch_result = data.get('esearchresult', {})
    id_list = esearch_result.get('idlist', [])
    
    if id_list:
        return {'pmid': id_list[0]}
    
    return None


def query_pubmed_all(dois, progress_callback=None):
    """
    Query PubMed for all DOIs.
    Returns dict mapping DOI -> identifier dict.
    
    Note: Only queries DOIs that don't already have a PMID from other sources.
    """
    results = {}
    total = len(dois)
    
    for i, doi in enumerate(dois):
        if progress_callback:
            progress_callback(i + 1, total, f"PubMed: {doi[:30]}...")
        
        result = query_pubmed(doi)
        if result:
            results[doi.lower()] = result
        
        # Rate limiting
        if i < total - 1:
            time.sleep(RATE_LIMITS['pubmed'])
    
    return results


print("✓ PubMed functions loaded")
print(f"  Rate limit: {RATE_LIMITS['pubmed']}s between requests")
print(f"  Returns: PubMed ID")

## QuickStatements Generation

*Functions to generate QuickStatements with proper source attribution.*

In [ ]:
def generate_quickstatements(enrichment_results):
    """
    Generate QuickStatements for adding identifiers to articles.
    
    Args:
        enrichment_results: List of dicts with:
            - qid: Wikidata QID
            - doi: DOI of article
            - identifiers: Dict of identifier_type -> value
            - sources: Dict of identifier_type -> source_name
    
    Returns:
        List of QuickStatements strings
    """
    statements = []
    today = datetime.now().strftime("+%Y-%m-%dT00:00:00Z/11")
    
    for item in enrichment_results:
        qid = item['qid']
        
        for id_type, value in item.get('identifiers', {}).items():
            if not value:
                continue
            
            prop = IDENTIFIER_PROPERTIES.get(id_type)
            if not prop:
                continue
            
            # Get source for this identifier
            source = item.get('sources', {}).get(id_type, 'openalex')
            source_qid = SOURCE_QIDS.get(source, SOURCE_QIDS['openalex'])
            
            # Format: QID|Property|"Value"|S248|SourceQID|S813|RetrievedDate
            # S248 = stated in, S813 = retrieved
            statement = f'{qid}|{prop}|"{value}"|S248|{source_qid}|S813|{today}'
            statements.append(statement)
    
    return statements


def format_quickstatements_output(statements):
    """
    Format QuickStatements for output/display.
    """
    if not statements:
        return "# No statements to generate"
    
    header = [
        "# Wikidata Article Identifier Enrichment",
        f"# Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"# Total statements: {len(statements)}",
        "#",
        "# Properties used:",
    ]
    
    for key, prop in IDENTIFIER_PROPERTIES.items():
        header.append(f"#   {prop}: {IDENTIFIER_NAMES[key]}")
    
    header.append("#")
    header.append("# Source references:")
    header.append("#   S248 = stated in")
    header.append("#   S813 = retrieved")
    header.append("")
    
    return "\n".join(header + statements)


print("✓ QuickStatements functions loaded")

## Master Enrichment Function

*Orchestrates the full enrichment pipeline across all APIs.*

In [ ]:
def enrich_articles(articles_df, apis_to_use=None, progress_output=None):
    """
    Enrich articles with identifiers from multiple APIs.
    
    Args:
        articles_df: DataFrame with columns: qid, doi, existing_* for each identifier
        apis_to_use: List of APIs to query ('openalex', 'semantic_scholar', 'fatcat', 'pubmed')
        progress_output: ipywidgets.Output for progress display
    
    Returns:
        List of enrichment results for QuickStatements generation
    """
    if apis_to_use is None:
        apis_to_use = ['openalex', 'semantic_scholar', 'fatcat', 'pubmed']
    
    # Get list of DOIs
    dois = articles_df['doi'].tolist()
    total_dois = len(dois)
    
    def log(msg):
        if progress_output:
            with progress_output:
                print(msg)
        else:
            print(msg)
    
    log(f"\n{'='*60}")
    log(f"Starting enrichment for {total_dois} articles")
    log(f"APIs to query: {', '.join(apis_to_use)}")
    log(f"{'='*60}\n")
    
    # Collect results from all APIs
    all_api_results = {}
    
    # Progress callback helper
    def make_progress_callback(api_name):
        def callback(current, total, msg):
            if progress_output:
                with progress_output:
                    clear_output(wait=True)
                    print(f"[{api_name}] {current}/{total}: {msg}")
        return callback
    
    # Query OpenAlex (batch capable)
    if 'openalex' in apis_to_use:
        log("📚 Querying OpenAlex...")
        start = time.time()
        openalex_results = query_openalex_all(
            dois,
            email=USER_EMAIL,
            progress_callback=make_progress_callback("OpenAlex")
        )
        all_api_results['openalex'] = openalex_results
        elapsed = time.time() - start
        log(f"  ✓ OpenAlex: {len(openalex_results)} results in {elapsed:.1f}s")
    
    # Query Semantic Scholar (one at a time, slow)
    if 'semantic_scholar' in apis_to_use:
        log("\n🔬 Querying Semantic Scholar (this takes a while)...")
        start = time.time()
        s2_results = query_semantic_scholar_all(
            dois,
            progress_callback=make_progress_callback("Semantic Scholar")
        )
        all_api_results['semantic_scholar'] = s2_results
        elapsed = time.time() - start
        log(f"  ✓ Semantic Scholar: {len(s2_results)} results in {elapsed:.1f}s")
    
    # Query Fatcat
    if 'fatcat' in apis_to_use:
        log("\n📁 Querying Fatcat...")
        start = time.time()
        fatcat_results = query_fatcat_all(
            dois,
            progress_callback=make_progress_callback("Fatcat")
        )
        all_api_results['fatcat'] = fatcat_results
        elapsed = time.time() - start
        log(f"  ✓ Fatcat: {len(fatcat_results)} results in {elapsed:.1f}s")
    
    # Query PubMed for DOIs that don't have PMID yet
    if 'pubmed' in apis_to_use:
        # Find DOIs without PMID from previous sources
        dois_needing_pmid = []
        for doi in dois:
            doi_lower = doi.lower()
            has_pmid = False
            
            # Check existing in Wikidata
            row = articles_df[articles_df['doi'].str.lower() == doi_lower]
            if len(row) > 0 and row.iloc[0].get('existing_pmid', ''):
                has_pmid = True
            
            # Check OpenAlex results
            if not has_pmid and 'openalex' in all_api_results:
                if doi_lower in all_api_results['openalex']:
                    if all_api_results['openalex'][doi_lower].get('pmid'):
                        has_pmid = True
            
            # Check Semantic Scholar results
            if not has_pmid and 'semantic_scholar' in all_api_results:
                if doi_lower in all_api_results['semantic_scholar']:
                    if all_api_results['semantic_scholar'][doi_lower].get('pmid'):
                        has_pmid = True
            
            if not has_pmid:
                dois_needing_pmid.append(doi)
        
        if dois_needing_pmid:
            log(f"\n🏥 Querying PubMed for {len(dois_needing_pmid)} DOIs without PMID...")
            start = time.time()
            pubmed_results = query_pubmed_all(
                dois_needing_pmid,
                progress_callback=make_progress_callback("PubMed")
            )
            all_api_results['pubmed'] = pubmed_results
            elapsed = time.time() - start
            log(f"  ✓ PubMed: {len(pubmed_results)} results in {elapsed:.1f}s")
        else:
            log("\n🏥 Skipping PubMed (all DOIs already have PMIDs)")
            all_api_results['pubmed'] = {}
    
    # Merge results and identify new identifiers to add
    log(f"\n{'='*60}")
    log("Merging results and identifying new identifiers...")
    log(f"{'='*60}\n")
    
    enrichment_results = []
    stats = defaultdict(int)
    
    for _, row in articles_df.iterrows():
        qid = row['qid']
        doi = row['doi']
        doi_lower = doi.lower()
        
        new_identifiers = {}
        sources = {}
        
        # Check each identifier type
        for id_type in IDENTIFIER_PROPERTIES.keys():
            # Skip if already exists in Wikidata
            existing = row.get(f'existing_{id_type}', '')
            if existing:
                stats[f'{id_type}_already_exists'] += 1
                continue
            
            # Look for this identifier in API results
            # Priority: OpenAlex > Semantic Scholar > PubMed > Fatcat
            value = None
            source = None
            
            # Check OpenAlex
            if 'openalex' in all_api_results and doi_lower in all_api_results['openalex']:
                v = all_api_results['openalex'][doi_lower].get(id_type)
                if v:
                    value = v
                    source = 'openalex'
            
            # Check Semantic Scholar (for S2 ID and PMID)
            if not value and 'semantic_scholar' in all_api_results and doi_lower in all_api_results['semantic_scholar']:
                v = all_api_results['semantic_scholar'][doi_lower].get(id_type)
                if v:
                    value = v
                    source = 'semantic_scholar'
            
            # Check PubMed (for PMID)
            if not value and 'pubmed' in all_api_results and doi_lower in all_api_results['pubmed']:
                v = all_api_results['pubmed'][doi_lower].get(id_type)
                if v:
                    value = v
                    source = 'pubmed'
            
            # Check Fatcat
            if not value and 'fatcat' in all_api_results and doi_lower in all_api_results['fatcat']:
                v = all_api_results['fatcat'][doi_lower].get(id_type)
                if v:
                    value = v
                    source = 'fatcat'
            
            if value:
                new_identifiers[id_type] = value
                sources[id_type] = source
                stats[f'{id_type}_found'] += 1
            else:
                stats[f'{id_type}_not_found'] += 1
        
        # Add to results if we found any new identifiers
        if new_identifiers:
            enrichment_results.append({
                'qid': qid,
                'doi': doi,
                'identifiers': new_identifiers,
                'sources': sources,
            })
    
    # Print summary
    log("\n📊 ENRICHMENT SUMMARY")
    log("-" * 40)
    
    for id_type, name in IDENTIFIER_NAMES.items():
        found = stats.get(f'{id_type}_found', 0)
        existing = stats.get(f'{id_type}_already_exists', 0)
        not_found = stats.get(f'{id_type}_not_found', 0)
        log(f"  {name}:")
        log(f"    Already in Wikidata: {existing}")
        log(f"    New to add: {found}")
        log(f"    Not available: {not_found}")
    
    log("-" * 40)
    log(f"Total articles: {total_dois}")
    log(f"Articles with new identifiers: {len(enrichment_results)}")
    
    return enrichment_results, stats


print("✓ Master enrichment function loaded")

---

# Interactive Interface

*Select input method and configure enrichment options.*

In [ ]:
# =============================================================================
# COMMON ANTHROPOLOGY JOURNALS (with Wikidata QIDs)
# =============================================================================
ANTHROPOLOGY_JOURNALS = {
    "American Anthropologist": "Q465636",
    "American Ethnologist": "Q4744138",
    "Cultural Anthropology": "Q5193569",
    "Journal of the Royal Anthropological Institute": "Q3186934",
    "Current Anthropology": "Q5194826",
    "Annual Review of Anthropology": "Q15763387",
    "American Journal of Physical Anthropology": "Q4744460",
    "Journal of Anthropological Archaeology": "Q6294134",
    "Anthropological Quarterly": "Q4772843",
    "Ethos": "Q15750622",
    "Medical Anthropology Quarterly": "Q6805643",
    "PoLAR: Political and Legal Anthropology Review": "Q7209596",
    "Anthropology & Education Quarterly": "Q15758498",
    "City & Society": "Q15752927",
    "Museum Anthropology": "Q15725476",
    "Anthropology of Work Review": "Q15762972",
    "Visual Anthropology Review": "Q15763055",
    "Nutritional Anthropology": "Q113519893",
    "Anthropology and Humanism": "Q15749991",
    "Journal of Linguistic Anthropology": "Q15752434",
    "Journal of Latin American and Caribbean Anthropology": "Q15752365",
    "Archaeology, Ethnology and Anthropology of Eurasia": "Q4098359",
}

# =============================================================================
# WIDGETS
# =============================================================================

# Input method selection
input_method = widgets.RadioButtons(
    options=['Select Journal(s)', 'Paste DOIs', 'Upload CSV'],
    value='Select Journal(s)',
    description='Input:',
    style={'description_width': 'initial'}
)

# Journal selection
journal_select = widgets.SelectMultiple(
    options=list(ANTHROPOLOGY_JOURNALS.keys()),
    value=[],
    description='Journals:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px', height='200px')
)

# DOI text area
doi_textarea = widgets.Textarea(
    placeholder='Enter DOIs (one per line or comma-separated)\n\nExample:\n10.1111/aman.13001\n10.1111/amet.12345',
    layout=widgets.Layout(width='400px', height='200px')
)

# File upload
file_upload = widgets.FileUpload(
    accept='.csv,.xlsx',
    multiple=False,
    description='Upload CSV'
)

# API selection
api_checkboxes = {
    'openalex': widgets.Checkbox(value=True, description='OpenAlex (fast, batch)'),
    'semantic_scholar': widgets.Checkbox(value=True, description='Semantic Scholar (slow)'),
    'fatcat': widgets.Checkbox(value=True, description='Fatcat'),
    'pubmed': widgets.Checkbox(value=True, description='PubMed (only if no PMID)'),
}

# Email input for OpenAlex
email_input = widgets.Text(
    value=USER_EMAIL,
    placeholder='your.email@example.com',
    description='Email (for OpenAlex):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Article limit
limit_slider = widgets.IntSlider(
    value=100,
    min=10,
    max=1000,
    step=10,
    description='Max articles:',
    style={'description_width': 'initial'}
)

# Run button
run_button = widgets.Button(
    description='🚀 Start Enrichment',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

# Output areas
progress_output = widgets.Output()
results_output = widgets.Output()

# =============================================================================
# LAYOUT
# =============================================================================

# Dynamic input area
input_container = widgets.VBox([journal_select])

def update_input_display(change):
    if change['new'] == 'Select Journal(s)':
        input_container.children = [journal_select]
    elif change['new'] == 'Paste DOIs':
        input_container.children = [doi_textarea]
    else:
        input_container.children = [file_upload]

input_method.observe(update_input_display, names='value')

# API checkboxes layout
api_box = widgets.VBox([
    widgets.HTML("<b>APIs to Query:</b>"),
    *api_checkboxes.values()
])

# Main layout
config_box = widgets.VBox([
    widgets.HTML(f"<div style='{CONTAINER_STYLE}'><h3 style='color:{COLORS["text_primary"]};margin-top:0'>⚙️ Configuration</h3></div>"),
    input_method,
    input_container,
    widgets.HTML("<hr>"),
    api_box,
    widgets.HTML("<hr>"),
    email_input,
    limit_slider,
    widgets.HTML("<br>"),
    run_button,
])

display(config_box)
display(progress_output)
display(results_output)

## Run Enrichment

*Execute the enrichment pipeline based on configuration.*

In [ ]:
# Global variables to store results
enrichment_results = []
quickstatements_text = ""
articles_df = None

def run_enrichment(b):
    global enrichment_results, quickstatements_text, articles_df, USER_EMAIL
    
    with progress_output:
        clear_output()
        print("Starting enrichment process...\n")
    
    with results_output:
        clear_output()
    
    # Update email
    USER_EMAIL = email_input.value
    
    # Get selected APIs
    apis_to_use = [api for api, cb in api_checkboxes.items() if cb.value]
    
    if not apis_to_use:
        with progress_output:
            print("❌ Error: Please select at least one API to query.")
        return
    
    # Get articles based on input method
    try:
        if input_method.value == 'Select Journal(s)':
            if not journal_select.value:
                with progress_output:
                    print("❌ Error: Please select at least one journal.")
                return
            
            with progress_output:
                print(f"Fetching articles from {len(journal_select.value)} journal(s)...")
            
            all_articles = []
            for journal_name in journal_select.value:
                journal_qid = ANTHROPOLOGY_JOURNALS[journal_name]
                with progress_output:
                    print(f"  Querying {journal_name} ({journal_qid})...")
                df = get_articles_by_journal(journal_qid, limit=limit_slider.value)
                if len(df) > 0:
                    df['journal'] = journal_name
                    all_articles.append(df)
                    with progress_output:
                        print(f"    Found {len(df)} articles")
                time.sleep(RATE_LIMITS['wikidata'])
            
            if not all_articles:
                with progress_output:
                    print("❌ No articles found in selected journals.")
                return
            
            articles_df = pd.concat(all_articles, ignore_index=True)
        
        elif input_method.value == 'Paste DOIs':
            if not doi_textarea.value.strip():
                with progress_output:
                    print("❌ Error: Please enter some DOIs.")
                return
            
            # Parse DOIs
            dois = extract_dois_from_text(doi_textarea.value)
            if not dois:
                with progress_output:
                    print("❌ Error: No valid DOIs found in input.")
                return
            
            with progress_output:
                print(f"Looking up {len(dois)} DOIs in Wikidata...")
            
            articles_df = batch_lookup_articles_by_doi(dois[:limit_slider.value])
            
            if len(articles_df) == 0:
                with progress_output:
                    print("❌ No articles found in Wikidata for the given DOIs.")
                return
        
        else:  # Upload CSV
            if not file_upload.value:
                with progress_output:
                    print("❌ Error: Please upload a CSV file.")
                return
            
            # Read uploaded file
            uploaded_file = list(file_upload.value.values())[0]
            content = uploaded_file['content']
            
            if uploaded_file['name'].endswith('.xlsx'):
                df = pd.read_excel(BytesIO(content))
            else:
                df = pd.read_csv(BytesIO(content))
            
            # Find DOI column
            doi_col = None
            for col in df.columns:
                if col.lower() in ['doi', 'DOI', 'Doi']:
                    doi_col = col
                    break
            
            if not doi_col:
                with progress_output:
                    print(f"❌ Error: No 'DOI' column found. Columns: {list(df.columns)}")
                return
            
            dois = df[doi_col].dropna().apply(clean_doi).dropna().tolist()
            
            with progress_output:
                print(f"Looking up {min(len(dois), limit_slider.value)} DOIs in Wikidata...")
            
            articles_df = batch_lookup_articles_by_doi(dois[:limit_slider.value])
            
            if len(articles_df) == 0:
                with progress_output:
                    print("❌ No articles found in Wikidata for the uploaded DOIs.")
                return
        
        with progress_output:
            print(f"\n✓ Found {len(articles_df)} articles in Wikidata")
        
        # Run enrichment
        enrichment_results, stats = enrich_articles(
            articles_df,
            apis_to_use=apis_to_use,
            progress_output=progress_output
        )
        
        # Generate QuickStatements
        statements = generate_quickstatements(enrichment_results)
        quickstatements_text = format_quickstatements_output(statements)
        
        # Display results
        with results_output:
            clear_output()
            
            print(f"\n{'='*60}")
            print("✅ ENRICHMENT COMPLETE")
            print(f"{'='*60}\n")
            
            print(f"Articles processed: {len(articles_df)}")
            print(f"Articles with new identifiers: {len(enrichment_results)}")
            print(f"QuickStatements generated: {len(statements)}")
            
            if statements:
                print("\n" + "-"*40)
                print("Preview (first 10 statements):")
                print("-"*40)
                for stmt in statements[:10]:
                    print(stmt)
                if len(statements) > 10:
                    print(f"... and {len(statements) - 10} more")
    
    except Exception as e:
        with progress_output:
            print(f"\n❌ Error: {e}")
            import traceback
            traceback.print_exc()


run_button.on_click(run_enrichment)
print("✓ Run button connected")

## Export Results

*Download QuickStatements and detailed results.*

In [ ]:
def export_quickstatements():
    """Export QuickStatements to file."""
    if not quickstatements_text:
        print("❌ No QuickStatements to export. Run enrichment first.")
        return
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"identifier_enrichment_quickstatements_{timestamp}.txt"
    
    with open(filename, 'w') as f:
        f.write(quickstatements_text)
    
    print(f"✓ Saved to {filename}")
    
    # Download in Colab
    try:
        from google.colab import files
        files.download(filename)
    except ImportError:
        print(f"  (Running outside Colab - file saved locally)")


def export_detailed_results():
    """Export detailed results to CSV."""
    if not enrichment_results:
        print("❌ No results to export. Run enrichment first.")
        return
    
    # Build detailed results
    rows = []
    for item in enrichment_results:
        row = {
            'qid': item['qid'],
            'doi': item['doi'],
        }
        for id_type in IDENTIFIER_PROPERTIES.keys():
            row[f'new_{id_type}'] = item.get('identifiers', {}).get(id_type, '')
            row[f'{id_type}_source'] = item.get('sources', {}).get(id_type, '')
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"identifier_enrichment_details_{timestamp}.csv"
    
    df.to_csv(filename, index=False)
    
    print(f"✓ Saved to {filename}")
    print(f"  Rows: {len(df)}")
    
    # Download in Colab
    try:
        from google.colab import files
        files.download(filename)
    except ImportError:
        print(f"  (Running outside Colab - file saved locally)")


def export_full_article_data():
    """Export full article data including existing identifiers."""
    if articles_df is None or len(articles_df) == 0:
        print("❌ No article data to export. Run enrichment first.")
        return
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"articles_with_identifiers_{timestamp}.csv"
    
    articles_df.to_csv(filename, index=False)
    
    print(f"✓ Saved to {filename}")
    print(f"  Rows: {len(articles_df)}")
    
    # Download in Colab
    try:
        from google.colab import files
        files.download(filename)
    except ImportError:
        print(f"  (Running outside Colab - file saved locally)")


# Export buttons
export_qs_btn = widgets.Button(description='📥 Download QuickStatements', button_style='success', layout=widgets.Layout(width='220px'))
export_details_btn = widgets.Button(description='📥 Download Details CSV', button_style='info', layout=widgets.Layout(width='220px'))
export_full_btn = widgets.Button(description='📥 Download Full Data', button_style='warning', layout=widgets.Layout(width='220px'))

export_output = widgets.Output()

def on_export_qs(b):
    with export_output:
        clear_output()
        export_quickstatements()

def on_export_details(b):
    with export_output:
        clear_output()
        export_detailed_results()

def on_export_full(b):
    with export_output:
        clear_output()
        export_full_article_data()

export_qs_btn.on_click(on_export_qs)
export_details_btn.on_click(on_export_details)
export_full_btn.on_click(on_export_full)

print("Export Options:")
display(widgets.HBox([export_qs_btn, export_details_btn, export_full_btn]))
display(export_output)

## Coverage Statistics

*Analyze identifier coverage across processed articles.*

In [ ]:
def show_coverage_stats():
    """Display coverage statistics for processed articles."""
    if articles_df is None or len(articles_df) == 0:
        print("❌ No data available. Run enrichment first.")
        return
    
    total = len(articles_df)
    
    print(f"\n{'='*60}")
    print("📊 IDENTIFIER COVERAGE ANALYSIS")
    print(f"{'='*60}\n")
    print(f"Total articles analyzed: {total}\n")
    
    print("Existing identifiers in Wikidata:")
    print("-" * 40)
    
    for id_type, name in IDENTIFIER_NAMES.items():
        col = f'existing_{id_type}'
        if col in articles_df.columns:
            has_id = articles_df[col].notna() & (articles_df[col] != '')
            count = has_id.sum()
            pct = (count / total * 100) if total > 0 else 0
            bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            print(f"  {name:25} {bar} {count:5} ({pct:5.1f}%)")
    
    if enrichment_results:
        print("\n" + "-" * 40)
        print("\nNew identifiers to add:")
        print("-" * 40)
        
        new_counts = defaultdict(int)
        for item in enrichment_results:
            for id_type in item.get('identifiers', {}).keys():
                new_counts[id_type] += 1
        
        for id_type, name in IDENTIFIER_NAMES.items():
            count = new_counts.get(id_type, 0)
            pct = (count / total * 100) if total > 0 else 0
            bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            print(f"  {name:25} {bar} {count:5} ({pct:5.1f}%)")
    
    print("\n" + "="*60)


# Stats button
stats_btn = widgets.Button(description='📊 Show Coverage Stats', button_style='info')
stats_output = widgets.Output()

def on_stats(b):
    with stats_output:
        clear_output()
        show_coverage_stats()

stats_btn.on_click(on_stats)

display(stats_btn)
display(stats_output)

---

## Quick Test

*Test the API connections with a single DOI.*

In [ ]:
def test_apis(test_doi="10.1111/aman.13001"):
    """
    Test all API connections with a single DOI.
    """
    print(f"Testing APIs with DOI: {test_doi}\n")
    print("="*50)
    
    # Test Wikidata
    print("\n1. Testing Wikidata lookup...")
    result = lookup_article_by_doi(test_doi)
    if result:
        print(f"   ✓ Found: {result['qid']}")
    else:
        print(f"   ⚠️ Not found in Wikidata (this is OK for testing)")
    
    # Test OpenAlex
    print("\n2. Testing OpenAlex...")
    oa_results = query_openalex_batch([test_doi])
    if test_doi.lower() in oa_results:
        print(f"   ✓ Found: {oa_results[test_doi.lower()]}")
    else:
        print(f"   ⚠️ Not found in OpenAlex")
    
    # Test Semantic Scholar
    print("\n3. Testing Semantic Scholar...")
    s2_result = query_semantic_scholar(test_doi)
    if s2_result:
        print(f"   ✓ Found: {s2_result}")
    else:
        print(f"   ⚠️ Not found in Semantic Scholar")
    
    # Test Fatcat
    print("\n4. Testing Fatcat...")
    fc_result = query_fatcat(test_doi)
    if fc_result:
        print(f"   ✓ Found: {fc_result}")
    else:
        print(f"   ⚠️ Not found in Fatcat")
    
    # Test PubMed
    print("\n5. Testing PubMed...")
    pm_result = query_pubmed(test_doi)
    if pm_result:
        print(f"   ✓ Found: {pm_result}")
    else:
        print(f"   ⚠️ Not found in PubMed (expected for most anthro articles)")
    
    print("\n" + "="*50)
    print("API testing complete!")


# Uncomment to run test:
# test_apis()

---

## Notes

### API Coverage

| API | Anthropology Coverage | Speed | Notes |
|-----|----------------------|-------|-------|
| OpenAlex | ~95% | Fast (batch) | Best single source |
| Semantic Scholar | ~70-80% | Slow | Weaker for qualitative work |
| Fatcat | ~60-70% | Medium | Open access focus |
| PubMed | ~5-15% | Fast | Mostly biomedical anthro |

### Rate Limits

The notebook uses conservative rate limiting to stay within free tier limits:

- **OpenAlex**: 1 second between batches (50 DOIs per batch)
- **Semantic Scholar**: 3.5 seconds between requests (100 req/5 min limit)
- **Fatcat**: 1 second between requests
- **PubMed**: 0.35 seconds between requests (3 req/sec limit)

### Time Estimates

For 100 articles with all APIs enabled:
- OpenAlex: ~3 seconds (2 batches)
- Semantic Scholar: ~6 minutes (100 × 3.5s)
- Fatcat: ~2 minutes (100 × 1s)
- PubMed: ~35 seconds (100 × 0.35s, only if needed)

**Total: ~9 minutes for 100 articles**

### Tips

1. Set your email in the configuration cell for OpenAlex's "polite pool" (faster limits)
2. Start with a small batch (10-20 articles) to test
3. Disable Semantic Scholar if you need faster results (it's the slowest)
4. PubMed is skipped automatically for DOIs that already have PMIDs